# News-enhanced model: A/B comparison

**Question:** does adding daily news sentiment as model features improve next-day return prediction?

**Design (fair A/B).** Both models share the same architecture, window, split and training settings. They differ only by the input features:
- **Base**: technical indicators only.
- **News**: technical indicators **+ 6 sentiment features** (`news_compound`, `news_pos`, `news_neg`, `news_neu`, `news_count`, `news_has`).

Both are trained on the 2021-2023 news window (yfinance prices), so the comparison is clean. The data is small (~450 days x 19 tickers), so we use a small model and a short context to avoid overfitting.

Expectation from notebook 14: news has a strong *same-day* effect but weak *next-day* predictability, so any improvement here should be **modest** — and that is itself a valid, honest result.

In [ ]:
import sys
from pathlib import Path

project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))
import os
os.chdir(project_root)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler

from src.data.preprocessor import preprocess_data
from src.data.feature_engineering import create_all_features, add_forward_log_return
from src.data.dataset import time_series_split, prepare_dataset
from src.data.news_sentiment import load_daily_sentiment, merge_sentiment_features, NEWS_FEATURE_COLUMNS
from src.data.pipeline import _exclude_columns_for_features
from src.models.transformer_model import StockTransformer
from src.training.trainer import Trainer
from src.utils.config import load_config, PROJECT_ROOT

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# Config overrides for this small-data experiment (do not touch the base model config).
config = load_config()
config.data.context_length = 20
config.data.prediction_horizon = 1
config.model.d_model = 64
config.model.n_heads = 4
config.model.n_layers = 2
config.model.d_ff = 128
config.training.num_epochs = 80
config.training.batch_size = 64
config.training.early_stopping.patience = 12

CTX = config.data.context_length
PRICE_COL = 'close'
TARGET_COL = 'log_return'
device = torch.device('cpu')
print('context length:', CTX, '| model d_model:', config.model.d_model, 'layers:', config.model.n_layers)

In [ ]:
def build_feature_frame(use_news: bool) -> tuple[pd.DataFrame, list[str]]:
    """Build a scaled feature frame from the news-window prices.

    Mirrors src/data/pipeline.extract_dataset but on the yfinance news-window
    prices, optionally merging sentiment features.
    """
    df = pd.read_parquet('data/raw/prices_news_window.parquet')
    df['date'] = pd.to_datetime(df['date'])

    df, _ = preprocess_data(
        df, handle_missing=True, missing_method='forward_fill',
        handle_outliers_flag=True, outliers_method='clip', normalize=False,
        date_column='date', symbol_column='symbol',
    )
    df = create_all_features(
        df, price_column=PRICE_COL, windows=config.data.features.windows,
        lags=[1, 2, 3, 5, 10] if config.data.features.lag_features else [],
        add_technical=config.data.features.technical_indicators,
        add_lags=config.data.features.lag_features,
        add_temporal=config.data.features.temporal_features,
        add_volume=True, simplified=config.data.features.simplified,
        symbol_column='symbol',
    )
    df = add_forward_log_return(df, price_column=PRICE_COL, target_column=TARGET_COL, symbol_column='symbol')

    if use_news:
        sentiment = load_daily_sentiment(config=config)
        df = merge_sentiment_features(df, sentiment, date_column='date', symbol_column='symbol')

    df = df.dropna().reset_index(drop=True)

    feature_columns = _exclude_columns_for_features(df, 'date', 'symbol', PRICE_COL, TARGET_COL)

    # Fit scaler on the per-symbol train portion only (no leakage).
    train_df, _, _ = time_series_split(
        df, config.data.train_split, config.data.val_split, config.data.test_split,
        'date', 'symbol', per_symbol=True,
    )
    scaler = StandardScaler().fit(train_df[feature_columns])
    df[feature_columns] = scaler.transform(df[feature_columns])
    return df, feature_columns


df_base, feat_base = build_feature_frame(use_news=False)
df_news, feat_news = build_feature_frame(use_news=True)

news_only = [c for c in feat_news if c in NEWS_FEATURE_COLUMNS]
print(f'Base features:  {len(feat_base)}')
print(f'News features:  {len(feat_news)}  (+{len(news_only)} sentiment: {news_only})')
print(f'Rows: base={len(df_base)}, news={len(df_news)}')
print('date range:', df_base['date'].min().date(), '->', df_base['date'].max().date())

In [ ]:
def make_datasets(df, feature_columns):
    return prepare_dataset(
        df, feature_columns=feature_columns, target_column=TARGET_COL,
        context_length=CTX, prediction_horizon=config.data.prediction_horizon,
        train_split=config.data.train_split, val_split=config.data.val_split,
        test_split=config.data.test_split, per_symbol=True,
    )


def build_model(input_dim):
    return StockTransformer(
        input_dim=input_dim, d_model=config.model.d_model, n_heads=config.model.n_heads,
        n_layers=config.model.n_layers, d_ff=config.model.d_ff, dropout=config.model.dropout,
        activation=config.model.activation, prediction_horizon=config.data.prediction_horizon,
    )


def train_variant(df, feature_columns, tag):
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    train_ds, val_ds, test_ds = make_datasets(df, feature_columns)
    train_loader = DataLoader(train_ds, batch_size=config.training.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=config.training.batch_size, shuffle=False, num_workers=0)

    model = build_model(len(feature_columns))
    config.paths.checkpoint_file = f'best_model_newsexp_{tag}.pt'
    trainer = Trainer(model, config, train_loader, val_loader, device=device)
    print(f'\n######## Training variant: {tag} ({len(feature_columns)} features) ########')
    history = trainer.train()
    return model, history, test_ds

In [ ]:
model_base, hist_base, test_base = train_variant(df_base, feat_base, 'base')

In [ ]:
model_news, hist_news, test_news = train_variant(df_news, feat_news, 'news')

In [ ]:
def evaluate(model, test_ds, tag):
    """Load best checkpoint and compute test metrics."""
    ckpt_path = PROJECT_ROOT / config.paths.models_dir / f'best_model_newsexp_{tag}.pt'
    ckpt = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=0)
    preds, tgts = [], []
    with torch.no_grad():
        for x, y in loader:
            out = model(x).cpu().numpy().reshape(-1)
            preds.append(out)
            tgts.append(y.numpy().reshape(-1))
    preds = np.concatenate(preds)
    tgts = np.concatenate(tgts)

    mae = np.mean(np.abs(preds - tgts))
    rmse = np.sqrt(np.mean((preds - tgts) ** 2))
    dir_acc = np.mean(np.sign(preds) == np.sign(tgts))
    corr = np.corrcoef(preds, tgts)[0, 1]
    return {
        'variant': tag, 'val_loss': ckpt.get('score'), 'test_MAE': mae,
        'test_RMSE': rmse, 'directional_acc': dir_acc, 'pred_vs_actual_corr': corr,
        'n_test': len(preds),
    }


res_base = evaluate(model_base, test_base, 'base')
res_news = evaluate(model_news, test_news, 'news')
summary = pd.DataFrame([res_base, res_news]).set_index('variant')
summary_display = summary.copy()
summary_display['directional_acc'] = (summary_display['directional_acc'] * 100).round(2).astype(str) + '%'
summary_display[['val_loss', 'test_MAE', 'test_RMSE', 'pred_vs_actual_corr']] = summary_display[['val_loss', 'test_MAE', 'test_RMSE', 'pred_vs_actual_corr']].round(6)
summary_display

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(hist_base['val_losses'], label='base', color='#f59e0b')
axes[0].plot(hist_news['val_losses'], label='news', color='#3b82f6')
axes[0].set_title('Validation loss')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss'); axes[0].legend()

metrics = ['directional_acc', 'pred_vs_actual_corr']
x = np.arange(len(metrics))
w = 0.35
axes[1].bar(x - w/2, [res_base[m] for m in metrics], w, label='base', color='#f59e0b')
axes[1].bar(x + w/2, [res_news[m] for m in metrics], w, label='news', color='#3b82f6')
axes[1].set_xticks(x); axes[1].set_xticklabels(metrics)
axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
axes[1].set_title('Test metrics: base vs news')
axes[1].legend()

plt.tight_layout()
plt.show()

delta = (res_news['directional_acc'] - res_base['directional_acc']) * 100
print(f"Directional accuracy: base={res_base['directional_acc']*100:.2f}%  news={res_news['directional_acc']*100:.2f}%  (delta={delta:+.2f} pp)")

## Summary

- Both models trained on the same 2021-2023 window and split; the only difference is the 6 sentiment features.
- The comparison quantifies the **marginal value of news** for next-day prediction.
- Given the efficient-market finding in notebook 14 (news is mostly priced in same-day), a small or mixed effect here is expected and scientifically honest — it shows we *measured* the contribution of news rather than assuming it.